In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('supply_chain_data.csv')

df['Total_Fulfillment_Cost'] = (df['Manufacturing costs'] * df['Production volumes']) + df['Costs']

df['Net_Profit'] = df['Revenue generated'] - df['Total_Fulfillment_Cost']
df['Profit_Margin_Pct'] = (df['Net_Profit'] / df['Revenue generated']) * 100
df['Inventory_Turnover_Ratio'] = df['Number of products sold'] / df['Stock levels'].replace(0, 1)

df['Total_Velocity_Days'] = df['Manufacturing lead time'] + df['Lead time'] + df['Shipping times']

df.to_csv('supply_analytics.csv', index=False)

print(df[['SKU', 'Net_Profit', 'Profit_Margin_Pct', 'Inventory_Turnover_Ratio', 'Total_Velocity_Days']].head())

    SKU    Net_Profit  Profit_Margin_Pct  Inventory_Turnover_Ratio  \
0  SKU0  -1475.929320         -17.039135                 13.827586   
1  SKU1 -10422.035063        -139.688710                 13.886792   
2  SKU2 -20362.237443        -212.599392                  8.000000   
3  SKU3 -25868.322423        -333.061249                  3.608696   
4  SKU4 -36351.911968       -1353.130179                174.200000   

   Total_Velocity_Days  
0                   62  
1                   55  
2                   41  
3                   48  
4                   16  


In [ ]:
import sqlite3
sql_df = df.copy()
sql_df.columns = [c.replace(' ', '_') for c in sql_df.columns]
conn = sqlite3.connect(':memory:')
sql_df.to_sql('supply_analytics', conn, index=False, if_exists='replace')

print("-- CRITICAL SQL QUERY 1: CARRIER QUALITY DEGRADATION REPORT ---")

query_carriers = """
SELECT
    Shipping_carriers,
    Transportation_modes,
    COUNT(SKU) as Total_Shipments_Tracked,
    ROUND(AVG(Defect_rates), 2) as Avg_Defect_Rate_Pct,
    ROUND(AVG(Shipping_costs), 2) as Avg_Shipping_Cost,
    ROUND(AVG(Total_Velocity_Days), 1) as Avg_End_To_End_Days
FROM supply_analytics
GROUP BY Shipping_carriers, Transportation_modes
ORDER BY Avg_Defect_Rate_Pct DESC;
"""

print(pd.read_sql_query(query_carriers, conn))


print("\n --- CRITICAL SQL QUERY 2: THE TOP 5 HIGHEST PROFIT-BLEED SKUs ---")

query_bleed = """
SELECT
    SKU,
    Product_type,
    Supplier_name,
    ROUND(Revenue_generated, 2) as Revenue,
    ROUND(Total_Fulfillment_Cost, 2) as Total_Overhead,
    ROUND(Net_Profit, 2) as Net_Loss,
    ROUND(Profit_Margin_Pct, 1) as Profit_Margin_Pct
FROM supply_analytics
WHERE Net_Profit < 0
ORDER BY Net_Profit ASC
LIMIT 5;
"""

print(pd.read_sql_query(query_bleed, conn))
conn.close()

-- CRITICAL SQL QUERY 1: CARRIER QUALITY DEGRADATION REPORT ---
   Shipping_carriers Transportation_modes  Total_Shipments_Tracked  \
0          Carrier C                 Rail                        6   
1          Carrier C                 Road                        7   
2          Carrier A                  Sea                        5   
3          Carrier A                  Air                        5   
4          Carrier A                 Road                       11   
5          Carrier A                 Rail                        7   
6          Carrier B                 Road                       11   
7          Carrier C                  Air                        7   
8          Carrier B                  Sea                        3   
9          Carrier C                  Sea                        9   
10         Carrier B                 Rail                       15   
11         Carrier B                  Air                       14   

    Avg_Defect_Rate_Pct  